In [5]:
!pip install gradio groq pandas


In [7]:
import os
import pandas as pd
import gradio as gr
import difflib
import traceback
from groq import Groq   # pip install groq

# -------- CONFIG --------
GROQ_API_KEY = os.environ.get("GROQ_API_KEY") or "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"
CSV_PATH = "ML_DL_Models_500.csv"
DEFAULT_MODEL = "llama-3.1-70b-versatile"  # safe Groq model
# ------------------------

# Load CSV
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"CSV not found: {CSV_PATH}")

df = pd.read_csv(CSV_PATH)
df.columns = [c.strip() for c in df.columns]

# ---- Auto-detect dataset column ----
DATASET_COL = None
for candidate in ["Dataset", "Datasets", "Task/Use Case"]:
    if candidate in df.columns:
        DATASET_COL = candidate
        break
if DATASET_COL is None:
    DATASET_COL = df.columns[0]  # fallback

print("Using dataset column:", DATASET_COL)

# Dataset names for fuzzy matching
dataset_names = df[DATASET_COL].dropna().astype(str).unique().tolist()

# Initialize Groq client
groq_client = Groq(api_key=GROQ_API_KEY)

# -------- Chatbot --------
def chat_fn(user_input: str, chosen_model: str, custom_model_id: str):
    if not user_input.strip():
        return "⚠️ Please write something."

    model_to_use = custom_model_id.strip() or chosen_model or DEFAULT_MODEL
    try:
        completion = groq_client.chat.completions.create(
            model=model_to_use,
            messages=[
                {"role": "system", "content": "You are a helpful AI assistant."},
                {"role": "user", "content": user_input}
            ],
            max_tokens=400,
        )
        return completion.choices[0].message.content
    except Exception as e:
        return f"⚠️ Error calling Groq API: {e}"

# -------- Recommender --------
def recommend_fn(dataset_query: str):
    if not dataset_query.strip():
        return "⚠️ Please enter a dataset name."

    q = dataset_query.strip()
    try:
        matches = df[df[DATASET_COL].astype(str).str.contains(q, case=False, na=False)]

        if matches.empty:
            close = difflib.get_close_matches(q, dataset_names, n=5, cutoff=0.4)
            if not close:
                return f"❌ No match found for '{q}'. Try another keyword."
            return "❌ No direct match. Did you mean:\n" + "\n".join(f"- {c}" for c in close)

        if "Accuracy" not in matches.columns:
            rows = matches.head(10)
            return "⚠️ Accuracy column missing.\n" + rows.to_string(index=False)

        tmp = matches.copy()
        tmp["Accuracy"] = pd.to_numeric(tmp["Accuracy"], errors="coerce")
        tmp = tmp.dropna(subset=["Accuracy"])
        if tmp.empty:
            return "⚠️ Matches found but no numeric Accuracy values."

        best = tmp.sort_values(by="Accuracy", ascending=False).iloc[0]
        return (f"✅ Best match for '{q}':\n"
                f"Model: {best.get('Model','-')}\n"
                f"Type: {best.get('Type','-')}\n"
                f"Dataset: {best.get(DATASET_COL,'-')}\n"
                f"Task: {best.get('Task','-')}\n"
                f"Accuracy: {best['Accuracy']}")
    except Exception as e:
        return f"⚠️ Error: {e}\n{traceback.format_exc()[:400]}"

# -------- Explorer --------
def explorer_fn(filter_text: str):
    if not filter_text.strip():
        return df.head(200)
    sub = df[df.apply(lambda r: r.astype(str).str.contains(filter_text, case=False, na=False).any(), axis=1)]
    if sub.empty:
        return f"No rows matching '{filter_text}'."
    return sub.head(500)

# -------- Gradio UI --------
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🌐 AI Model Hub — Chatbot + Recommender + Explorer (Groq LLM)")

    with gr.Tabs():
        with gr.TabItem("Chatbot"):
            with gr.Row():
                model_dropdown = gr.Dropdown(
                    choices=[
                        "llama-3.1-8b-instant",
                        "llama-3.1-70b-versatile",
                        "gemma2-9b-it",
                        "gemma2-27b-it",
                    ],
                    value="llama-3.1-70b-versatile",
                    label="Select Groq LLM"
                )
                custom_model = gr.Textbox(label="Custom Model ID", placeholder="Optional: enter Groq model id")
            chat_in = gr.Textbox(lines=3, label="User Prompt")
            chat_out = gr.Textbox(lines=10, label="Response")
            chat_btn = gr.Button("Send")
            chat_btn.click(chat_fn, inputs=[chat_in, model_dropdown, custom_model], outputs=chat_out)

        with gr.TabItem("Recommender"):
            q = gr.Textbox(label="Dataset Search", placeholder="e.g., imagenet, squad, skin cancer")
            rec_out = gr.Textbox(lines=6, label="Recommendation")
            rec_btn = gr.Button("Recommend")
            rec_btn.click(recommend_fn, inputs=q, outputs=rec_out)

        with gr.TabItem("Explorer"):
            filter_box = gr.Textbox(label="Filter keyword", placeholder="model name, dataset, task...")
            df_out = gr.Dataframe(value=df.head(100), label="Models Table")
            show_btn = gr.Button("Apply Filter")
            show_btn.click(explorer_fn, inputs=filter_box, outputs=df_out)

if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", share=True)


Using dataset column: Task/Use Case
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c2ac8665b1ec8cabd5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
